# Digit Recognition (MNIST)

This notebook demonstrates training a neural network on the classic MNIST handwritten digit dataset.

**Dataset**: 70,000 images of handwritten digits (0-9), 28x28 pixels each.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')

from neura_command import (
    Sequential, Dense, ReLU, Softmax, Dropout,
    Adam, CrossEntropyLoss, EarlyStopping,
)

np.random.seed(42)

## 1. Load and Prepare Data

We'll use scikit-learn to fetch MNIST.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Fetch MNIST (this may take a moment)
print("Loading MNIST dataset...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='liac-arff')

X = mnist.data.astype(np.float32)
y = mnist.target.astype(np.int32)

print(f"Dataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Unique labels: {np.unique(y)}")

In [ ]:
# Normalize pixel values to [0, 1]
X = X / 255.0

# Convert labels to one-hot encoding
n_classes = 10
y_onehot = np.eye(n_classes)[y].astype(np.float32)

print(f"X range: [{X.min():.2f}, {X.max():.2f}]")
print(f"One-hot shape: {y_onehot.shape}")

In [ ]:
# Split into train/validation/test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_onehot, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# Keep original labels for test set
y_test_labels = np.argmax(y_test, axis=1)

print(f"Training samples:   {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")
print(f"Test samples:       {len(X_test):,}")

In [ ]:
# Visualize some examples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = np.random.randint(len(X_train))
    ax.imshow(X_train[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f"Label: {np.argmax(y_train[idx])}")
    ax.axis('off')
plt.suptitle('Sample MNIST Digits', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Create the Model

We'll use a simple feedforward network:
- Input: 784 pixels (28×28 flattened)
- Hidden: 256 → 128 units with ReLU and Dropout
- Output: 10 classes with Softmax

In [ ]:
model = Sequential(
    layers=[
        Dense(784, 256),
        ReLU(),
        Dropout(0.2),
        Dense(256, 128),
        ReLU(),
        Dropout(0.2),
        Dense(128, 10),
        Softmax(),
    ],
    optimizer=Adam(learning_rate=0.001),
    loss=CrossEntropyLoss(),
)

print(model.summary())

## 3. Train the Model

In [ ]:
# Use a smaller subset for faster training in this demo
# Remove these lines for full training
train_subset = 10000
X_train_subset = X_train[:train_subset]
y_train_subset = y_train[:train_subset]

print(f"Training on {len(X_train_subset):,} samples...")

In [ ]:
# Train with early stopping
history = model.train(
    X_train_subset, y_train_subset,
    epochs=20,
    batch_size=64,
    validation_data=(X_val[:2000], y_val[:2000]),
    callbacks=[EarlyStopping(patience=3, min_delta=0.001)],
    verbose=True,
)

In [ ]:
# Plot training history
plt.figure(figsize=(10, 4))
plt.plot(history['loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training History')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Evaluate the Model

In [ ]:
# Make predictions on test set
predictions = model.predict(X_test)
predicted_labels = np.argmax(predictions, axis=1)

# Calculate accuracy
accuracy = np.mean(predicted_labels == y_test_labels) * 100
print(f"Test Accuracy: {accuracy:.2f}%")

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_labels, predicted_labels)

plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')

# Add text annotations
for i in range(10):
    for j in range(10):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')

plt.xticks(range(10))
plt.yticks(range(10))
plt.show()

In [ ]:
# Per-class accuracy
print("Per-class accuracy:")
print("-" * 30)
for digit in range(10):
    mask = y_test_labels == digit
    digit_accuracy = np.mean(predicted_labels[mask] == digit) * 100
    print(f"Digit {digit}: {digit_accuracy:.1f}%")

## 5. Visualize Predictions

In [ ]:
# Show some correct predictions
correct_mask = predicted_labels == y_test_labels
correct_indices = np.where(correct_mask)[0]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = correct_indices[np.random.randint(len(correct_indices))]
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f"Pred: {predicted_labels[idx]}", color='green')
    ax.axis('off')
plt.suptitle('Correct Predictions ✓', fontsize=14, color='green')
plt.tight_layout()
plt.show()

In [ ]:
# Show some incorrect predictions
incorrect_indices = np.where(~correct_mask)[0]

if len(incorrect_indices) > 0:
    n_show = min(10, len(incorrect_indices))
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i, ax in enumerate(axes.flat):
        if i < len(incorrect_indices):
            idx = incorrect_indices[i]
            ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
            ax.set_title(f"Pred: {predicted_labels[idx]}, True: {y_test_labels[idx]}", color='red')
        ax.axis('off')
    plt.suptitle('Incorrect Predictions ✗', fontsize=14, color='red')
    plt.tight_layout()
    plt.show()
else:
    print("No incorrect predictions!")

## 6. Save the Model

In [ ]:
import tempfile
import os

# Save the trained model
with tempfile.TemporaryDirectory() as tmpdir:
    model_path = os.path.join(tmpdir, 'mnist_model')
    model.save(model_path)
    
    # Check file sizes
    json_size = os.path.getsize(model_path + '.json')
    npz_size = os.path.getsize(model_path + '.npz')
    
    print(f"Model saved!")
    print(f"  Architecture (JSON): {json_size / 1024:.1f} KB")
    print(f"  Weights (NPZ):       {npz_size / 1024:.1f} KB")
    print(f"  Total:               {(json_size + npz_size) / 1024:.1f} KB")

## 7. Interactive Prediction (Optional)

Draw a digit and see the model predict it!

In [ ]:
def predict_single(model, image):
    """Predict a single image and show probabilities."""
    # Reshape and normalize if needed
    if image.shape != (1, 784):
        image = image.reshape(1, 784)
    if image.max() > 1:
        image = image / 255.0
    
    # Get prediction
    probs = model.predict(image.astype(np.float32))[0]
    predicted = np.argmax(probs)
    
    # Display
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    
    ax1.imshow(image.reshape(28, 28), cmap='gray')
    ax1.set_title(f"Predicted: {predicted}", fontsize=16)
    ax1.axis('off')
    
    colors = ['green' if i == predicted else 'steelblue' for i in range(10)]
    ax2.bar(range(10), probs, color=colors)
    ax2.set_xlabel('Digit')
    ax2.set_ylabel('Probability')
    ax2.set_title('Prediction Probabilities')
    ax2.set_xticks(range(10))
    
    plt.tight_layout()
    plt.show()
    
    return predicted, probs

# Test with a random image
test_idx = np.random.randint(len(X_test))
pred, probs = predict_single(model, X_test[test_idx])
print(f"True label: {y_test_labels[test_idx]}")

## Summary

We trained a simple neural network on MNIST achieving good accuracy:

| Aspect | Value |
|--------|-------|
| Architecture | 784 → 256 → 128 → 10 |
| Parameters | ~235,000 |
| Training time | ~1-2 minutes |
| Test accuracy | ~95-97% |

**Tips for better accuracy:**
- Train on full dataset (49,000 samples)
- More epochs
- Data augmentation
- Convolutional layers (not yet in NeuraCommand)